### Space Counts and Rates

This dataset is pretty clean. Covers from 6 mar 2017 to 19 Jan 2018

With this dataset, we will know:

Which area it refers to: zone

When the record/snapshot was created
* as_of: date the dataset entry reflects

How much parking supply is there
* spaces: number of parking spaces in that zone
* meters: number of meters in that zone

How much it costs
* rate: numeric hourly rate
* rate_description: text version of the rate

What kind of parking
* type: usually on-street vs off-street

In [1]:
import polars as pl

space_counts_rates = pl.read_csv("raw_data/Space Counts and Rates.csv")
space_counts_rates.head(3)

_id,zone,as_of,spaces,rate,rate_description,meters,type
i64,str,str,i64,f64,str,i64,str
1,"""301 - Sheridan Harvard Lot""","""2017-03-06""",41,1.0,"""$1.00/HR""",2,"""off-street"""
2,"""302 - Sheridan Kirkwood Lot""","""2017-03-06""",114,1.0,"""$1.00/HR""",3,"""off-street"""
3,"""304 - Tamello Beatty Lot""","""2017-03-06""",76,1.0,"""$1.00/HR""",2,"""off-street"""


In [2]:
space_counts_rates.describe()

statistic,_id,zone,as_of,spaces,rate,rate_description,meters,type
str,f64,str,str,f64,f64,str,f64,str
"""count""",125.0,"""125""","""125""",125.0,121.0,"""125""",119.0,"""125"""
"""null_count""",0.0,"""0""","""0""",0.0,4.0,"""0""",6.0,"""0"""
"""mean""",68.976,null,null,169.552,1.359504,null,18.663866,null
"""std""",49.589194,null,null,191.030628,0.764371,null,25.958662,null
"""min""",1.0,"""301 - Sheridan Harvard Lot""","""2017-03-06""",12.0,0.5,"""$0.50/HR""",1.0,"""off-street"""
"""25%""",32.0,null,null,41.0,1.0,null,2.0,null
"""50%""",63.0,null,null,76.0,1.0,null,3.0,null
"""75%""",94.0,null,null,237.0,1.5,null,31.0,null
"""max""",218.0,"""W CIRC DR""","""2018-01-19""",757.0,4.0,"""*SPECIAL""",101.0,"""on-street"""


**Ufficial Data Dictionary**

https://data.wprdc.org/tl/dataset/zone-and-lot-attributes/resource/4d4a0668-7905-4db6-985e-f7575a9e3160

| Column             | Type   | Label            | Description                            |
| ------------------ | ------ | ---------------- | -------------------------------------- |
| `zone`             | text   | Zone             | Reporting-zone identifier              |
| `as_of`            | text   | As of            | Date the dataset was created.          |
| `spaces`           | int4   | Space count      | Number of parking spaces in zone       |
| `rate`             | float8 | Rate             | The cost per hour to park at the meter |
| `rate_description` | text   | Rate description | Hourly cost of parking in zone (text)  |
| `meters`           | int4   | Meter count      | Number of meters in zone               |
| `type`             | text   |                  |                                        |


**Time Range Analysis**

In [3]:
# Transform as_of to polars datetime
space_counts_rates_transformed = space_counts_rates.with_columns(
    pl.col("as_of")
    .str.to_datetime(format="%Y-%m-%d", strict=False)
    .dt.date()
)
space_counts_rates_transformed.head(3)

_id,zone,as_of,spaces,rate,rate_description,meters,type
i64,str,date,i64,f64,str,i64,str
1,"""301 - Sheridan Harvard Lot""",2017-03-06,41,1.0,"""$1.00/HR""",2,"""off-street"""
2,"""302 - Sheridan Kirkwood Lot""",2017-03-06,114,1.0,"""$1.00/HR""",3,"""off-street"""
3,"""304 - Tamello Beatty Lot""",2017-03-06,76,1.0,"""$1.00/HR""",2,"""off-street"""


In [4]:
# Check unique value of rate_description to know whether all rate refer to hourly rate
space_counts_rates.select("rate_description").unique()

rate_description
str
"""$1.50($2 after 2pm)/HR"""
"""$4.00/HR"""
"""$2.00/HR"""
"""$0.50/HR"""
"""$3.00/HR"""
"""*SPECIAL"""
"""$1.50/HR"""
"""$1/HR"""
"""$1.00/HR"""


In [5]:
# Check time range of the data
print(f"Earliest date in space counts and rates data: \
    {space_counts_rates_transformed.select('as_of').min()}")
print(f"Latest date in space counts and rates data: \
    {space_counts_rates_transformed.select('as_of').max()}")

Earliest date in space counts and rates data:     shape: (1, 1)
┌────────────┐
│ as_of      │
│ ---        │
│ date       │
╞════════════╡
│ 2017-03-06 │
└────────────┘
Latest date in space counts and rates data:     shape: (1, 1)
┌────────────┐
│ as_of      │
│ ---        │
│ date       │
╞════════════╡
│ 2018-01-19 │
└────────────┘


**Print unique value of some columns**

In [6]:
space_counts_rates_transformed.select("type").unique()

type
str
"""off-street"""
"""on-street"""


### Data analysis

**There are multiple rows with same zone!**   
See following code:

In [7]:
# Check unique values in space_counts_rates_transformed.zone
unique_zones = space_counts_rates_transformed.select("zone").sort(by="zone").unique().to_series().to_list()
print("Number of rows in s:", space_counts_rates_transformed.height)
print("Number of unique zones in s.zone:", len(unique_zones))
print(unique_zones)

Number of rows in s: 125
Number of unique zones in s.zone: 67
['301 - Sheridan Harvard Lot', '302 - Sheridan Kirkwood Lot', '304 - Tamello Beatty Lot', '307 - Eva Beatty Lot', '311 - Ansley Beatty Lot', '314 - Penn Circle NW Lot', '321 - Beacon Bartlett Lot', '322 - Forbes Shady Lot', '323 - Douglas Phillips Lot', '324 - Forbes Murray Lot', '325 - JCC/Forbes Lot', '328 - Ivy Bellefonte Lot', '331 - Homewood Zenith Lot', '334 - Taylor Street Lot', '335 - Friendship Cedarville Lot', '337 - 52nd & Butler Lot', '338 - 42nd & Butler Lot', '341 - 18th & Sidney Lot', '342 - East Carson Lot', '343 - 19th & Carson Lot', '344 - 18th & Carson Lot', '345 - 20th & Sidney Lot', '351 - Brownsville & Sandkey Lot', '354 - Walter/Warrington Lot', '355 - Asteroid Warrington Lot', '357 - Shiloh Street Lot', '361 - Brookline Lot', '363 - Beechview Lot', '369 - Main/Alexander Lot', '371 - East Ohio Street Lot', '375 - Oberservatory Hill Lot', '401 - Downtown 1', '402 - Downtown 2', '403 - Uptown', '404 - St

**We should take one row for every zone -> which one?**  

I would say take the latest one.   

The existance of duplicate allows us to do analysis like from old date
to new date is there any changes in terms of rate, number of meters etc

In [8]:
# We should take one row for every zone -> which one?
# find duplicated zone
dup_zones = space_counts_rates_transformed.group_by("zone").len().filter(pl.col("len") > 1)
space_counts_rates_transformed.join(dup_zones.select("zone"), on="zone", how="inner").sort("zone")

_id,zone,as_of,spaces,rate,rate_description,meters,type
i64,str,date,i64,f64,str,i64,str
1,"""301 - Sheridan Harvard Lot""",2017-03-06,41,1.0,"""$1.00/HR""",2,"""off-street"""
57,"""301 - Sheridan Harvard Lot""",2018-01-19,41,1.0,"""$1.00/HR""",2,"""off-street"""
2,"""302 - Sheridan Kirkwood Lot""",2017-03-06,114,1.0,"""$1.00/HR""",3,"""off-street"""
58,"""302 - Sheridan Kirkwood Lot""",2018-01-19,114,1.0,"""$1.00/HR""",3,"""off-street"""
3,"""304 - Tamello Beatty Lot""",2017-03-06,76,1.0,"""$1.00/HR""",2,"""off-street"""
…,…,…,…,…,…,…,…
111,"""424 - Technology Drive""",2018-01-19,48,null,"""*SPECIAL""",5,"""on-street"""
56,"""425 - Bakery Sq""",2017-03-06,43,2.0,"""$2.00/HR""",4,"""on-street"""
112,"""425 - Bakery Sq""",2018-01-19,45,2.0,"""$2.00/HR""",4,"""on-street"""


In [9]:
# Take the latest one record
latest_per_zone = (
    space_counts_rates
    .with_columns(pl.col("as_of"))
    .sort(["zone", "as_of"], descending=[False, True])
    .unique(subset=["zone"], keep="first")
)
print(f"number of rows after removing the duplicate: {latest_per_zone.height}")
latest_per_zone.head()

number of rows after removing the duplicate: 67


_id,zone,as_of,spaces,rate,rate_description,meters,type
i64,str,str,i64,f64,str,i64,str
57,"""301 - Sheridan Harvard Lot""","""2018-01-19""",41,1.0,"""$1.00/HR""",2,"""off-street"""
58,"""302 - Sheridan Kirkwood Lot""","""2018-01-19""",114,1.0,"""$1.00/HR""",3,"""off-street"""
59,"""304 - Tamello Beatty Lot""","""2018-01-19""",76,1.0,"""$1.00/HR""",2,"""off-street"""
60,"""307 - Eva Beatty Lot""","""2018-01-19""",130,1.0,"""$1.00/HR""",2,"""off-street"""
61,"""311 - Ansley Beatty Lot""","""2018-01-19""",23,1.0,"""$1.00/HR""",1,"""off-street"""


data year analysis

In [10]:
# data year analysis
year_analysis = latest_per_zone.with_columns(
    pl.col("as_of").str.strptime(pl.Date, format="%Y-%m-%d", strict=False).alias("date")
)
year_analysis

_id,zone,as_of,spaces,rate,rate_description,meters,type,date
i64,str,str,i64,f64,str,i64,str,date
57,"""301 - Sheridan Harvard Lot""","""2018-01-19""",41,1.0,"""$1.00/HR""",2,"""off-street""",2018-01-19
58,"""302 - Sheridan Kirkwood Lot""","""2018-01-19""",114,1.0,"""$1.00/HR""",3,"""off-street""",2018-01-19
59,"""304 - Tamello Beatty Lot""","""2018-01-19""",76,1.0,"""$1.00/HR""",2,"""off-street""",2018-01-19
60,"""307 - Eva Beatty Lot""","""2018-01-19""",130,1.0,"""$1.00/HR""",2,"""off-street""",2018-01-19
61,"""311 - Ansley Beatty Lot""","""2018-01-19""",23,1.0,"""$1.00/HR""",1,"""off-street""",2018-01-19
…,…,…,…,…,…,…,…,…
215,"""SQ.HILL2""","""2017-11-13""",80,1.5,"""$1.50/HR""",null,"""on-street""",2017-11-13
218,"""Southside Lots""","""2018-01-19""",228,1.0,"""$1.00/HR""",null,"""off-street""",2018-01-19
216,"""UPTOWN1""","""2017-11-28""",261,1.5,"""$1.50/HR""",null,"""on-street""",2017-11-28


In [11]:
year_analysis.with_columns(
    pl.col("date").dt.year().alias("year")
).select("year").unique()

year
i32
2018
2017


**Save cleaned dataset**

In [12]:
# save the cleaned data
latest_per_zone.drop("_id").write_csv("../cleaned_data/space_counts_rates_cleaned.csv")